In [ ]:
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

X, y = datasets.load_iris(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2, 
    random_state=42
)



# Entrenar el modelo
params = {
    "random_state": 42,
    "max_iter": 1000
}
# Usamos **params para 
# desempaquetar el diccionario como argumentos
model = LogisticRegression(**params)
model.fit(X_train, y_train)
    
# Predecir en el set de prueba
y_pred = model.predict(X_test)

# Calcular métricas de evaluación
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print(f"✅ Accuracy: {accuracy:.4f}")
print(f"✅ Precision: {precision:.4f}")
print(f"✅ Recall: {recall:.4f}")
print(f"✅ F1: {f1:.4f}")

✅ Accuracy: 1.0000
✅ Precision: 1.0000
✅ Recall: 1.0000
✅ F1: 1.0000


In [ ]:
import mlflow
import pandas as pd
from mlflow.models import infer_signature

# mlflow server --host 0.0.0.0 --port 5000 
# en la terminal para iniciar el servidor de MLflow
mlflow.set_tracking_uri(uri="http://localhost:5000")  

mlflow.set_experiment(experiment_name="iris-classifier")

with mlflow.start_run():
    mlflow.log_params(params)
    
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1", f1)
    
    mlflow.set_tag("Training info", "Iris dataset classification with Logistic Regression")
    
    # Predecir en el set de entreno
    y_sol = model.predict(X_train)      
    signatures = infer_signature(X_train, y_sol)
    
    # logging the model
    model_info = mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="iris_classifier",
        signature=signatures,
        input_example=X_train,
        registered_model_name="tracking-quickstart-iris-classifier"
    )
    
    # cargar el modelo
    # loaded_model = mlflow.pyfunc.load_model(model_info.model_uri)
    loaded_model = mlflow.sklearn.load_model(model_uri=model_info.model_uri)
    
    predictions = loaded_model.predict(X_test)
    
    iris_feature_names = datasets.load_iris().feature_names
    
    result = pd.DataFrame(X_test, columns=iris_feature_names)
    result["actual_class"] = y_test
    result["predicted_class"] = predictions
    
    result[:4]

2026/07/10 18:50:06 INFO mlflow.tracking.fluent: Experiment with name 'iris-classifier' does not exist. Creating a new experiment.


Successfully registered model 'tracking-quickstart-iris-classifier'.
2026/07/10 18:50:10 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: tracking-quickstart-iris-classifier, version 1
Created version '1' of model 'tracking-quickstart-iris-classifier'.
/home/renato/Desktop/MLOps-practicas/azure_mlflow_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm

🏃 View run classy-fox-256 at: http://localhost:5000/#/experiments/618702986255311960/runs/19a003a8b34e4889aa9fc3068d5cbc3a
🧪 View experiment at: http://localhost:5000/#/experiments/618702986255311960
